# Passing structured invoice data with `context=` in Deep Agents

This notebook demonstrates how to pass structured business data (invoice IDs) into a Deep Agent **without** embedding them in the user message.

**Pattern:** define `context_schema=InvoiceContext` at agent creation, pass `context={"invoice_ids": [...]}` at invoke time. Tools read scoped IDs from `runtime.context`.

Context is immutable for the run and is **not** automatically added to the model prompt — ideal for tenant IDs, scoped resource lists, credentials, and feature flags.

Put `OPENAI_API_KEY` in a `.env` file in this directory (see `.env.example`). Notebook kernels do not see terminal exports.

In [ ]:
from pprint import pprint

from deepagents import create_deep_agent
from langchain.chat_models import init_chat_model

from invoice_context_demo.demo_support import (
    InvoiceContext,
    build_analyze_invoice_tool,
    build_fake_invoices,
    pretty_print_messages,
    require_openai_api_key,
)

require_openai_api_key()

MODEL = init_chat_model("openai:gpt-4.1-mini", temperature=0)

INVOICE_REGISTRY = build_fake_invoices()
INVOICE_IDS = list(INVOICE_REGISTRY)
USER_MESSAGE = "Please analyze the attached invoices."

print("Fake invoices in registry:")
for invoice_id, record in INVOICE_REGISTRY.items():
    print(f"  {invoice_id}: {record.vendor} (${record.amount_usd:,.2f}, {record.status})")

## Run the agent

The user message stays generic. Invoice IDs are passed in the structured `context=` payload. The tool reads allowed IDs from `runtime.context`.

In [ ]:
analyze_invoice = build_analyze_invoice_tool(INVOICE_REGISTRY)

agent = create_deep_agent(
    model=MODEL,
    tools=[analyze_invoice],
    context_schema=InvoiceContext,
    system_prompt=(
        "You analyze invoices using the analyze_invoice tool. "
        f"Scoped invoice IDs for this run (not in the user message): {', '.join(INVOICE_IDS)}. "
        "Call analyze_invoice once per scoped ID, then summarize the results."
    ),
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": USER_MESSAGE}]},
    config={"configurable": {"thread_id": "invoice-demo-context"}},
    context={"invoice_ids": INVOICE_IDS},
)

print("User message (no invoice IDs embedded):")
print(f"  {USER_MESSAGE}\n")
print("Structured payload passed via context=")
pprint({"invoice_ids": INVOICE_IDS})
print("\nAgent run:")
pretty_print_messages(result)

## Related patterns

| Mechanism | Use for |
|-----------|---------|
| **`context=`** (this demo) | Immutable per-run metadata: tenant IDs, scoped resource IDs, API keys |
| **`state_schema` + invoke keys** | Data that should checkpoint with the thread or change during the conversation |
| **`config`** | Thread IDs and tracing only — not business payload |
| **`files=`** | Binary attachments (PDFs, images) via StateBackend or FilesystemBackend |

Docs: [Context engineering in Deep Agents](https://docs.langchain.com/oss/python/deepagents/context-engineering)